In [122]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel
import torch

In [123]:
#read Dataset
org_df = pd.read_csv('D:/CIS545_Proj/dataset/Airbnb_Open_Data.csv')

C:\Users\asus\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3185: DtypeWarning: Columns (25) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [124]:
pd.set_option('display.max_columns', None)
org_df.head(5)

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,country code,instant_bookable,cancellation_policy,room type,Construction year,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020.0,$966,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,False,moderate,Entire home/apt,2007.0,$142,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,US,True,flexible,Private room,2005.0,$620,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,US,True,moderate,Entire home/apt,2005.0,$368,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,US,False,moderate,Entire home/apt,2009.0,$204,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [125]:
org_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

In [126]:
#drop unnecessary features
cl_drop = ['id','NAME','host id','host name','country','country code','last review','reviews per month', 'license']
df_droped = org_df.drop(columns=cl_drop)
df_droped = df_droped.dropna()

In [127]:
#convert obj to float
df_droped['price'] = df_droped['price'].replace('[\$,]', '', regex=True).astype(float)
df_droped['service fee'] = df_droped['service fee'].replace('[\$,]', '', regex=True).astype(float)

In [128]:
#convert obj to numerical categories
cl_en = ['host_identity_verified', 'instant_bookable', 'cancellation_policy']
encoder = LabelEncoder()
for column in cl_en:
    if df_droped[column].dtype == 'object':
        df_droped[column] = encoder.fit_transform(df_droped[column])
        print(f"\nMapping of '{column}' categories to encoded values:")
        for original_category, encoded_value in zip(encoder.classes_, encoder.transform(encoder.classes_)):
            print(f"{original_category}: {encoded_value}")


Mapping of 'host_identity_verified' categories to encoded values:
unconfirmed: 0
verified: 1

Mapping of 'instant_bookable' categories to encoded values:
False: 0
True: 1

Mapping of 'cancellation_policy' categories to encoded values:
flexible: 0
moderate: 1
strict: 2


In [129]:
df_droped.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 49200 entries, 0 to 102595
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   host_identity_verified          49200 non-null  int32  
 1   neighbourhood group             49200 non-null  object 
 2   neighbourhood                   49200 non-null  object 
 3   lat                             49200 non-null  float64
 4   long                            49200 non-null  float64
 5   instant_bookable                49200 non-null  int32  
 6   cancellation_policy             49200 non-null  int32  
 7   room type                       49200 non-null  object 
 8   Construction year               49200 non-null  float64
 9   price                           49200 non-null  float64
 10  service fee                     49200 non-null  float64
 11  minimum nights                  49200 non-null  float64
 12  number of reviews              

In [130]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
def get_word_embedding(word):
    inputs = tokenizer(word, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        word_embedding = outputs.pooler_output.squeeze().cpu().numpy()
    return word_embedding

cl_em = ['neighbourhood group','neighbourhood','room type','house_rules']
for column in cl_em:
    df_droped[column] = df_droped[column].apply(get_word_embedding)

df_droped.head(5)

C:\Users\asus\anaconda3\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


KeyboardInterrupt: 